In [0]:
# Register the winning run in Unity Catalog, alias it `challenger`, and move `champion`
# only when validation RMSE improves by at least `min_improvement` (relative).
import sys
import os
sys.path.insert(0, os.getcwd())

import mlflow
from mlflow import MlflowClient
from utils import (load_project, model_name as qualified_model_name,
                   classifier_model_name as qualified_classifier_name, widget_or)

mlflow.set_registry_uri("databricks-uc")
cfg = load_project()

dbutils.widgets.text("catalog", "stock_demo")
dbutils.widgets.text("min_improvement", "")     # empty -> project.yml
dbutils.widgets.text("run_id", "")              # optional: register this MLflow run (lets this task be re-run on its own)
catalog = dbutils.widgets.get("catalog")
min_improvement = float(widget_or(dbutils.widgets.get("min_improvement"),
                                  cfg["model"]["min_validation_rmse_improvement"]))

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.models")

# Get the winning model's MLflow run ID
run_id = dbutils.widgets.get("run_id").strip() or dbutils.jobs.taskValues.get(taskKey="train_models", key="run_id")
print(f"Registering model from run: {run_id}")

registered_model_name = qualified_model_name(cfg, catalog)
mv = mlflow.register_model(model_uri=f"runs:/{run_id}/model", name=registered_model_name)
print(f"Registered model version: {mv.version}")

client = MlflowClient()
client.set_registered_model_alias(registered_model_name, "challenger", str(mv.version))
print(f"Alias 'challenger' now points to version {mv.version}")

metrics = client.get_run(run_id).data.metrics
new_rmse = float(metrics["val_rmse"])
for k in ("val_rmse", "test_rmse", "test_directional_accuracy", "test_persistence_rmse"):
    client.set_model_version_tag(registered_model_name, str(mv.version), k, f"{float(metrics[k]):.6f}")

# --- Promotion check: does the new model beat the current champion on validation RMSE? ---
try:
    champion_mv = client.get_model_version_by_alias(registered_model_name, "champion")
    champion_rmse = float(client.get_run(champion_mv.run_id).data.metrics["val_rmse"])
    print(f"Current champion: v{champion_mv.version}, val_rmse={champion_rmse:.6f}")
except Exception:
    champion_rmse = float("inf")   # no champion alias yet: first run always promotes
    print("No existing champion found - this run will become champion by default.")

print(f"Challenger: v{mv.version}, val_rmse={new_rmse:.6f}")
promoted = new_rmse < champion_rmse * (1.0 - min_improvement)
if promoted:
    client.set_registered_model_alias(registered_model_name, "champion", str(mv.version))
    print(f"Promoted: 'champion' now points to version {mv.version}")
else:
    print("Not promoted - challenger did not beat the current champion.")

dbutils.jobs.taskValues.set(key="promoted", value=bool(promoted))
dbutils.jobs.taskValues.set(key="model_version", value=str(mv.version))


# Register and independently promote the best UP/DOWN classifier. This model is not
# connected to the existing price-regression serving endpoint.
classifier_run_id = dbutils.jobs.taskValues.get(taskKey="train_models", key="best_classifier_run_id")
classifier_registered_name = qualified_classifier_name(cfg, catalog)
classifier_mv = mlflow.register_model(model_uri=f"runs:/{classifier_run_id}/model",
                                      name=classifier_registered_name)
client.set_registered_model_alias(classifier_registered_name, "challenger", str(classifier_mv.version))
classifier_metrics = client.get_run(classifier_run_id).data.metrics
classifier_val_score = float(classifier_metrics["val_balanced_accuracy"])
classifier_test_score = float(classifier_metrics["test_balanced_accuracy"])
classifier_min_test_score = float(cfg["evaluation"].get("min_classifier_test_balanced_accuracy", 0.50))
for key in ("val_accuracy", "val_balanced_accuracy", "test_accuracy", "test_balanced_accuracy", "test_roc_auc"):
    if key in classifier_metrics:
        client.set_model_version_tag(classifier_registered_name, str(classifier_mv.version),
                                     key, f"{float(classifier_metrics[key]):.6f}")
try:
    classifier_champion = client.get_model_version_by_alias(classifier_registered_name, "champion")
    old_classifier_score = float(client.get_run(classifier_champion.run_id).data.metrics["val_balanced_accuracy"])
except Exception:
    old_classifier_score = float("-inf")
classifier_promoted = (classifier_val_score > old_classifier_score and
                        classifier_test_score >= classifier_min_test_score)
if classifier_promoted:
    client.set_registered_model_alias(classifier_registered_name, "champion", str(classifier_mv.version))
    print(f"Direction classifier champion -> {classifier_registered_name} v{classifier_mv.version} "
          f"(validation balanced accuracy={classifier_val_score:.4f})")
else:
    print(f"Direction classifier challenger retained; validation balanced accuracy={classifier_val_score:.4f}, "
          f"test balanced accuracy={classifier_test_score:.4f} (minimum {classifier_min_test_score:.4f})")

dbutils.jobs.taskValues.set(key="classifier_promoted", value=bool(classifier_promoted))
dbutils.jobs.taskValues.set(key="classifier_model_version", value=str(classifier_mv.version))